# Python과 가상환경

현재 Python 버전을 확인하고, 서로 구분되는 두 개의 가상환경을 임시 폴더에 만듭니다. 인터프리터는 Python 코드를 읽어 실행하는 프로그램이고, 가상환경은 프로젝트별 Python 실행 위치와 패키지 설치 위치를 구분합니다. 노트북에서는 선택한 커널이 코드를 실행합니다.

## Python 3.12 확인

가상환경을 만들기 전에 현재 커널이 교재의 기준 버전인지 확인합니다. `import sys`는 실행 중인 Python 정보를 읽는 표준 기능을 가져옵니다. `version_info[:2]`는 버전 정보의 앞 두 값인 주 버전과 부 버전을 고르고, `== (3, 12)`는 두 값이 3과 12인지 비교합니다. 결과가 `False`이면 오른쪽 위에서 Python 3.12 커널을 다시 선택합니다.

In [1]:
import sys

# version_info[:2]는 주 버전과 부 버전 두 개만 선택합니다.
version_matches = sys.version_info[:2] == (3, 12)
print(f"Python 3.12: {version_matches}")
assert version_matches

Python 3.12: True


## 임시 가상환경 두 개 생성

`uv venv`는 지정한 폴더에 Python 가상환경을 만듭니다. `--python 3.12`는 Python 버전을 고정하고, `--no-project`는 현재 프로젝트 설정을 읽지 않고 지정한 폴더에 환경만 만들게 합니다. `shutil.which("uv")`는 실행 가능한 uv 명령의 위치를 찾아 경로를 반환하며, 찾지 못하면 `None`을 반환합니다. `subprocess.run`의 대괄호 안에는 터미널에서 실행할 프로그램과 인자를 순서대로 적습니다. `check=True`는 명령 실패 시 실행을 멈춥니다. 이 실습에서는 추가 패키지를 설치하지 않습니다.

`uv is not None`은 명령 위치가 없는 상태인지 확인하는 식입니다. `None`이 아니면 참입니다. `assert 조건, "안내 문장"`의 쉼표 뒤 문자열은 조건이 거짓일 때 오류에 표시할 설명입니다.

In [2]:
import shutil
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory

# uv로 임시 폴더 안에 env_a와 env_b를 만듭니다.
uv = shutil.which("uv")
assert uv is not None, "uv 설치를 먼저 확인하세요."
temporary_folder = TemporaryDirectory()
root = Path(temporary_folder.name)
env_a = root / "env_a"
env_b = root / "env_b"
subprocess.run([uv, "-q", "venv", "--python", "3.12", "--no-project", env_a], check=True)
subprocess.run([uv, "-q", "venv", "--python", "3.12", "--no-project", env_b], check=True)
print("가상환경 2개 생성")

가상환경 2개 생성


## 각 환경의 Python 실행

운영체제에 맞는 Python 실행 파일을 고른 뒤, 각 환경 안에서 `prefix`가 기본 Python의 `base_prefix`와 다른지 확인합니다. 조건문의 한 줄 표현은 Windows이면 `Scripts/python.exe`, 그렇지 않으면 `bin/python`을 선택합니다. `subprocess.run`의 `-c`는 뒤의 Python 코드를 파일 없이 실행하는 인자입니다. `capture_output=True`는 별도 실행의 출력을 반환 결과의 `stdout`에 저장하고, `text=True`는 그 출력을 글자로 읽습니다. `splitlines()`는 여러 출력 줄을 줄별 목록으로 나눕니다.

In [3]:
import os

# Windows와 macOS/Linux의 Python 실행 파일 위치가 다를 수 있어 분기합니다.
python_name = Path("Scripts/python.exe") if os.name == "nt" else Path("bin/python")
check_code = "import sys; print(sys.prefix != sys.base_prefix); print(sys.prefix)"
result_a = subprocess.run([env_a / python_name, "-c", check_code], capture_output=True, text=True, check=True)
result_b = subprocess.run([env_b / python_name, "-c", check_code], capture_output=True, text=True, check=True)
lines_a = result_a.stdout.splitlines()
lines_b = result_b.stdout.splitlines()
print(f"env_a 가상환경: {lines_a[0]}")
print(f"env_b 가상환경: {lines_b[0]}")

env_a 가상환경: True
env_b 가상환경: True


## 서로 구분되는지 확인

각 Python이 보고하는 가상환경 위치(`sys.prefix`)가 다르면 각각 구분된 환경입니다. `lines_a[0]`은 첫 번째 결과인 가상환경 여부이고, `lines_a[1]`은 두 번째 결과인 환경 위치입니다. 환경 A와 B의 위치가 다른지, 두 실행이 모두 가상환경 안인지 차례로 확인합니다. 개인 경로는 출력하지 않고 비교 결과만 표시합니다.

마지막 `and`는 양쪽 조건이 모두 참일 때 참이 됩니다. 환경 위치가 서로 다르고 두 실행이 모두 가상환경 안에서 이루어져야 검사를 통과합니다.

In [4]:
# 각 실행의 sys.prefix가 서로 다른 환경 폴더를 가리키는지 확인합니다.
different_environments = lines_a[1] != lines_b[1]
both_inside = lines_a[0] == lines_b[0] == "True"
print(f"서로 다른 환경: {different_environments}")
print(f"두 실행 모두 가상환경 내부: {both_inside}")
assert different_environments and both_inside
temporary_folder.cleanup()

서로 다른 환경: True
두 실행 모두 가상환경 내부: True


## 실험 결과 해석

두 환경은 같은 Python 3.12를 기반으로 만들었지만 각 환경의 `sys.prefix`는 서로 다릅니다. 이 결과는 환경의 실행 위치가 구분되었음을 보여 줍니다. 제삼자 패키지를 설치하지 않았으므로 패키지 버전 충돌을 재현했다고 해석하지 않습니다. 교재 프로젝트를 준비할 때는 `uv sync --locked`로 잠금 파일에 맞는 환경을 만듭니다.